# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`src/Conformance/Corpus.ts`, mirrored statement for statement in Python)
and commits its snapshots under `conformance/<implementation>/`. This suite checks this implementation's files are
current, that its JSON is byte-identical to every other implementation's, and that it reads every other
implementation's JSON and YAML back to the same terms. Regenerate with `npm run conformance`. The corpus has a case for
Patterns and for every dialect, each a term written with its dialect, then its kind; each case validates in its own
dialect, within a holder in the corpus's store, after `Patterns.define` and each dialect's `define`, C's and C++'s within
a holder naming the ABI their types are read under, x86-64's System V ABI. The cases that are
data, not terms, are dialects' singletons, each written as one document: `domains`, ISO/IEC 11404's dialect, its
domains, owned by name, each kind its dialect's then its own, an enum and a packed one among them; and each dialect's
stored domains (`systemverilog_domains`, `python3_domains`, `typescript5_domains`, `matlab_domains`, `excel_domains`,
`c_domains`, `cpp_domains`), and each ABI dialect's (`amd64_domains`, `aapcs64_domains`); their domains validate, and
each holds the editions of the standards its dialect covers, by designation. One more is data, an edition, written as
one document: `standards`, C23's, ISO/IEC 9899:2024, its formal reference's parts, its common name and the kinds it
has. One is invalid syntax, `invalid`: a Python3 term holding an
`error`, a `missing` term and literals with their spelling, one of them holding no value but its problem; its problems
are reported.

In [ ]:
import * as Y from "yaml";
import { readdirSync, readFileSync, statSync } from "node:fs";

import { CASES, DIALECTS, DOMAINS, STANDARD, build } from "../src/Conformance/Corpus.js";
import { render } from "../src/Conformance/write.js";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import { Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import { Domains as FD, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { JSON, Plain, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, same } from "./support.js";

const ROOT = "../../conformance";
const OWN = "typescript5";
const OTHERS = readdirSync(ROOT).filter((n) => n !== OWN && statSync(`${ROOT}/${n}`).isDirectory()).sort();
assert(same(OTHERS, ["python3"]), OTHERS.join());

const corpus = build();
const snapshots = new Map([...corpus].map(([name, [schema, root, store]]) => [name, Plain.ToPlain(store)(schema, root)]));
const yaml11 = (t: string) => Y.parse(t, { version: "1.1", intAsBigInt: true, mapAsMap: true });

<a id="CONF-01"></a>
## CONF-01 · This implementation's committed files are current

In [ ]:
for (const [name, text] of render(corpus)) {
  assert(readFileSync(`${ROOT}/${OWN}/${name}`, "utf8") === text, `${name} is stale: regenerate the corpus`);
}

<a id="CONF-02"></a>
## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
{
  const expected = CASES.flatMap((c) => [`${c}.json`, `${c}.yaml`]).sort();
  for (const name of [OWN, ...OTHERS]) assert(same(readdirSync(`${ROOT}/${name}`).sort(), expected), name);
  for (const other of OTHERS) {
    for (const name of CASES) {
      const mine = readFileSync(`${ROOT}/${OWN}/${name}.json`);
      const theirs = readFileSync(`${ROOT}/${other}/${name}.json`);
      assert(mine.equals(theirs), `${name}.json differs from ${other}`);
    }
  }
}

<a id="CONF-03"></a>
## CONF-03 · Every implementation's YAML reads back to exactly this implementation's snapshot, also under YAML 1.1

In [ ]:
for (const other of [OWN, ...OTHERS]) {
  for (const name of CASES) {
    const text = readFileSync(`${ROOT}/${other}/${name}.yaml`, "utf8");
    assert(equal(YAML.loads(text), snapshots.get(name)), `${other}/${name}.yaml`);
    assert(equal(yaml11(text), snapshots.get(name)), `${other}/${name}.yaml under YAML 1.1`);
  }
}

<a id="CONF-04"></a>
## CONF-04 · Every implementation's JSON and YAML deserialize into this implementation's store, and validate

In [ ]:
const [ISO, SV] = ["isoiec11404", "systemverilog"];
const STORED = new Map<string, [string, [string, string]][]>([ // each case that is data: its dialect's domains, in order, by name, each its kind, its dialect's tag then its own
  ["domains", [["boolean", [ISO, "boolean"]], ["integer", [ISO, "integer"]], ["octetstring", [ISO, "octetstring"]],
    ["octet", [ISO, "octet"]], ["modulo(256)", [ISO, "modulo"]], ["state", [ISO, "state"]], ["mode", [ISO, "packed"]]]],
  ["systemverilog_domains", [...["logic", "reg", "integer", "time"].map((name): [string, [string, string]] => [name, [SV, "logic"]]),
    ...["bit", "byte", "shortint", "int", "longint"].map((name): [string, [string, string]] => [name, [SV, "bit"]]),
    ["real", ["ieee754", "ieee754"]], ["shortreal", ["ieee754", "ieee754"]]]],
  ["python3_domains", [["bool", [ISO, "boolean"]], ["int", [ISO, "integer"]], ["float", ["ieee754", "ieee754"]],
    ["str", ["python3", "str"]], ["bytes", [ISO, "octetstring"]], ["NoneType", ["python3", "nonetype"]],
    ["ellipsis", ["python3", "ellipsis"]]]],
  ["typescript5_domains", [["boolean", [ISO, "boolean"]], ["number", ["ieee754", "ieee754"]], ["bigint", [ISO, "integer"]],
    ["string", ["typescript5", "string"]], ["null", ["typescript5", "null"]]]],
  ["matlab_domains", [["double", ["ieee754", "ieee754"]], ["logical", [ISO, "boolean"]], ["string", ["matlab", "string"]]]],
  ["excel_domains", [["number", ["ieee754", "ieee754"]], ["logical", [ISO, "boolean"]], ["text", ["excel", "text"]],
    ["error", ["excel", "error"]]]],
  ["c_domains", [["bool", [ISO, "boolean"]], ...[8, 16, 32, 64].map((w): [string, [string, string]] => [`int${w}_t`, [ISO, "integer"]]),
    ...[8, 16, 32, 64].map((w): [string, [string, string]] => [`uint${w}_t`, [ISO, "modulo"]]),
    ...["_Float16", "_Float32", "_Float64", "_Float128", "_Decimal32", "_Decimal64", "_Decimal128"].map(
      (name): [string, [string, string]] => [name, ["ieee754", "ieee754"]]), ["nullptr_t", ["c", "nullptr_t"]]]],
  ["cpp_domains", [["bool", [ISO, "boolean"]], ...[8, 16, 32, 64].map((w): [string, [string, string]] => [`int${w}_t`, [ISO, "integer"]]),
    ...[8, 16, 32, 64].map((w): [string, [string, string]] => [`uint${w}_t`, [ISO, "modulo"]]),
    ...["float16_t", "float32_t", "float64_t", "float128_t", "bfloat16_t"].map(
      (name): [string, [string, string]] => [`std::${name}`, ["ieee754", "ieee754"]]), ["nullptr_t", ["cpp", "nullptr_t"]]]],
  ...([["amd64_domains", "integer", "integer"], ["aapcs64_domains", "modulo", "modulo"]] as [string, string, string][]).map(
    ([name, char, wchar]): [string, [string, [string, string]][]] => [name, [ // each ABI's, as it fixes them: char signed or not, wchar_t too
      ...([["char", char], ["signed char", "integer"], ["unsigned char", "modulo"], ["short", "integer"],
        ["unsigned short", "modulo"], ["int", "integer"], ["unsigned int", "modulo"], ["long", "integer"],
        ["unsigned long", "modulo"], ["long long", "integer"], ["unsigned long long", "modulo"]] as [string, string][]).map(
        ([domain, kind]): [string, [string, string]] => [domain, [ISO, kind]]),
      ...["float", "double", "long double"].map((domain): [string, [string, string]] => [domain, ["ieee754", "ieee754"]]),
      ["wchar_t", [ISO, wchar]], ...["char8_t", "char16_t", "char32_t", "size_t"].map((domain): [string, [string, string]] => [domain, [ISO, "modulo"]]),
      ...["ptrdiff_t", "intptr_t"].map((domain): [string, [string, string]] => [domain, [ISO, "integer"]]), ["uintptr_t", [ISO, "modulo"]],
      ...[8, 16, 32, 64].map((w): [string, [string, string]] => [`int_least${w}_t`, [ISO, "integer"]]),
      ...[8, 16, 32, 64].map((w): [string, [string, string]] => [`uint_least${w}_t`, [ISO, "modulo"]]),
      ["_Float32x", ["ieee754", "ieee754"]], ["_Float64x", ["ieee754", "ieee754"]]]]),
]);
assert(same([...STORED.keys()], [...DOMAINS.keys()]));
const ISO_IEC = ["org.iso", "ch.iec"];
const COVERED = new Map<string, Map<string, unknown>>([ // each data case's dialect's editions, by designation, each its
  // formal reference's parts and common names
  ["domains", new Map([["ISO/IEC 11404:2007", map({ bodies: ISO_IEC, number: "11404", revision: ["year", 2007n] })]])],
  ["systemverilog_domains", new Map(([["1364", "Verilog", [1995n, 2001n, 2005n]], ["1800", "SystemVerilog", [2005n, 2009n, 2012n, 2017n, 2023n]]] as
    [string, string, bigint[]][]).flatMap(([number, name, years]) => years.map((year) => [`IEEE Std ${number}-${year}`,
    map({ bodies: ["org.ieee"], number, revision: ["year", year], names: [`${name}-${year}`] })]))) as Map<string, unknown>],
  ["python3_domains", new Map(Array.from({ length: 16 }, (_, m) => [`Python 3.${m}`,
    map({ bodies: ["org.python"], revision: ["version", `3.${m}`] })]))],
  ["typescript5_domains", new Map(Array.from({ length: 10 }, (_, m) => [`TypeScript 5.${m}`,
    map({ bodies: ["com.microsoft"], revision: ["version", `5.${m}`] })]))],
  ["matlab_domains", new Map()], ["excel_domains", new Map()],
  ["c_domains", new Map(([[2011n, "C11"], [2018n, "C17"], [2024n, "C23"]] as [bigint, string][]).map(([year, name]) => [
    `ISO/IEC 9899:${year}`, map({ bodies: ISO_IEC, number: "9899", revision: ["year", year], names: [name] })]))],
  ["cpp_domains", new Map(([[2017n, "C++17"], [2020n, "C++20"], [2024n, "C++23"]] as [bigint, string][]).map(([year, name]) => [
    `ISO/IEC 14882:${year}`, map({ bodies: ISO_IEC, number: "14882", revision: ["year", year], names: [name] })]))],
  ["amd64_domains", new Map([["System V Application Binary Interface, AMD64 Architecture Processor Supplement", map({})]])],
  ["aapcs64_domains", new Map([["Procedure Call Standard for the Arm 64-bit Architecture (AArch64)",
    map({ bodies: ["com.arm"], names: ["AAPCS64"] })]])],
]);
assert(same([...COVERED.keys()], [...DOMAINS.keys()]) && same([...STANDARD.keys()], ["standards"]));
for (const store of new Set([...corpus.values()].map(([, , store]) => store))) {
  store.transaction(Patterns.define); // Patterns' operations, which validation reads from a term's holder's store
}
for (const other of [OWN, ...OTHERS]) {
  for (const [name, [schema, root, store]] of corpus) {
    const dialect = DIALECTS.get(name) ?? E.DIALECT;
    for (const [suffix, decode] of [["json", JSON.FromJSON(store)], ["yaml", YAML.FromYAML(store)]] as [string, any][]) {
      const restored = decode(schema, new Uint8Array(readFileSync(`${ROOT}/${other}/${name}.${suffix}`)));
      assert(equal(Plain.ToPlain(store)(schema, restored), snapshots.get(name)), `${other}/${name}.${suffix}`);
      if (DOMAINS.has(name)) { // a dialect, a singleton: read back into itself, its domains valid
        assert(restored === root && Validators.Validate(store)(schema, restored).length === 0);
        const kinds = STORED.get(name) as [string, [string, string]][];
        assert(same([...restored.domains].map((e: any) => `${e.name}`), kinds.map(([domain]) => domain)) && [...restored.operations].length === 0);
        assert(equal([...restored.domains].map((e: any) => [FD.nameOf(e.domain), FD.kindOf(e.domain).slice(0, 2)]), kinds));
        assert(equal(new Map([...FS.standards(restored)].map(([d, s]) => [d, FS.properties(s)])), COVERED.get(name)));
        continue;
      }
      if (STANDARD.has(name)) { // an edition its dialect owns: read back into itself, valid, with its kinds
        assert(restored === root && Validators.Validate(store)(schema, restored).length === 0);
        assert(FS.designationOf(restored) === "ISO/IEC 9899:2024" && same(FS.tags(restored), ["ISO/IEC 9899:2024", "C23"]));
        assert(equal(FS.properties(restored), COVERED.get("c_domains")!.get("ISO/IEC 9899:2024")));
        assert(same(FS.kinds(restored).slice(0, 2).map((k: any) => k.name), ["Expressions.C.Expressions.Literal",
          "Expressions.C.Expressions.Constant"]));
        assert(FS.kinds(restored).length === 21); // C's kinds but GNU's two extensions and invalid syntax
        continue;
      }
      assert(F.same(restored, root), `${other}/${name}.${suffix}`);
      // validated within a holder, in the corpus's store; C's and C++'s naming their ABI
      const held = store.transaction(() => F.term(store.Expressions.Framework.Expression((e: any) => (["c", "cpp"].includes(name)
        ? e.root(F.spec(restored)).abi([store.Instances.Expressions.SystemV.AMD64]) : e.root(F.spec(restored))))));
      const problems = dialect.validate(held, SY.free(held, dialect));
      if (name === "invalid") { // invalid syntax and a literal's problem, kept and reported
        assert(same(problems, ["left: left: right: invalid syntax '(x +* 1)'",
          "left: right: leading zeros in a decimal integer literal are not permitted", "right: missing expression"]), other);
        continue;
      }
      assert(problems.length === 0, `${other}/${name}.${suffix}`);
    }
  }
}